# Fama-French 3 Factor Backtest Demo

This notebook displays the three FF3 factor return series, turnover, winsorized decile layer tests, and IC/R2 diagnostics for single cross-sectional signals.

Decile layers are built by formation date: winsorize the signal at 1%/99%, split stocks into 10 buckets, value-weight within each bucket, and hold to the next annual rebalance window.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT.parent))
sys.path.insert(0, str(PROJECT / 'code'))

import pandas as pd
import plotly.graph_objects as go
from factors import FactorBacktester

RESULTS = PROJECT / 'results'
DEMO = RESULTS / 'backtest_demo'
bt = FactorBacktester(periods_per_year=12)
cols = ['ann_return', 'ann_vol', 'sharpe', 'total_return', 'max_drawdown', 'hit_rate', 'avg_turnover']
ic_cols = ['mean_ic', 'mean_rank_ic', 'mean_cos_ic', 'ic_ir', 'rank_ic_ir', 'cos_ic_ir', 'mean_r2']
DEMO


## Three FF3 Factor Returns

These are the monthly factor return series from our FF3 construction. `avg_turnover` is computed from the investable holdings/weights used to represent each factor.

In [ ]:
factor_summary = pd.read_csv(DEMO / 'ff3_factor_summary.csv', index_col=0)
factor_summary[cols].style.format('{:.4f}')


In [ ]:
ff3 = pd.read_csv(RESULTS / 'ff3_factors_2017_2025.csv', parse_dates=['date'])
fig = go.Figure()
for name in ['MKT_RF', 'SMB', 'HML']:
    r = ff3[['date', name]].dropna().copy()
    r['cum_return'] = (1 + r[name]).cumprod() - 1
    fig.add_trace(go.Scatter(x=r['date'].to_numpy(dtype='datetime64[ns]'), y=r['cum_return'], name=name))
fig.update_yaxes(tickformat='.2%')
fig.update_layout(template='plotly_white', height=560, title='FF3 factor cumulative returns')
fig.show()


In [ ]:
for name in ['MKT_RF', 'SMB', 'HML']:
    port = ff3[['date', name]].rename(columns={name: 'ret'})
    fig = bt.plot_performance(port)
    fig.update_layout(title=f'{name} performance and drawdown')
    fig.show()


## Winsorized Decile Layers

The market factor is not a single stock-level characteristic, so there is no natural cross-sectional decile split for `MKT_RF`.

For the cross-sectional parts of FF3, the decile tests below use `BM` for value and `market_equity` for size. `D01` is the lowest signal bucket and `D10` is the highest signal bucket.

In [ ]:
spread_summary = pd.read_csv(DEMO / 'winsor_decile_long_short_summary.csv', index_col=0)
spread_summary[cols].style.format('{:.4f}')


## IC / R2 Diagnostics

IC is computed cross-sectionally at each June formation date between the winsorized signal and next-month stock return. The table includes Pearson IC, Rank IC, Cos IC, their IR-style ratios, and average cross-sectional R2.

In [ ]:
ic_summary = pd.read_csv(DEMO / 'winsor_signal_ic_r2_summary.csv', index_col=0)
ic_summary[ic_cols].style.format('{:.4f}')


### BM Deciles

`D01` is low book-to-market, `D10` is high book-to-market. The spread file uses `D10 - D01`.

In [ ]:
bm_summary = pd.read_csv(DEMO / 'bm_winsor_decile_summary.csv', index_col=0)
bm_summary[cols[:-1]].style.format('{:.4f}')


In [ ]:
bm_layers = pd.read_csv(DEMO / 'bm_winsor_decile_layer_returns.csv', parse_dates=['date'])
fig = bt.plot_layers(bm_layers)
fig.update_layout(title='BM winsorized decile layer returns')
fig.show()


In [ ]:
bm_spread = pd.read_csv(DEMO / 'bm_winsor_decile_long_short.csv', parse_dates=['date'])
fig = bt.plot_performance(bm_spread)
fig.update_layout(title='BM decile spread: D10 - D01')
fig.show()


In [ ]:
bm_ic = pd.read_csv(DEMO / 'bm_winsor_ic.csv', parse_dates=['date'])
fig = go.Figure()
for col in ['cum_ic', 'cum_rank_ic', 'cum_cos_ic']:
    fig.add_trace(go.Scatter(x=bm_ic['date'].to_numpy(dtype='datetime64[ns]'), y=bm_ic[col], name=col))
fig.update_layout(template='plotly_white', height=500, title='BM cumulative IC')
fig.show()
display(bm_ic[['date', 'ic', 'rank_ic', 'cos_ic', 'n_assets']].tail())


### Size / ME Deciles

`D01` is small market equity, `D10` is large market equity. To match the SMB direction, the spread file uses `D01 - D10`.

In [ ]:
size_summary = pd.read_csv(DEMO / 'size_me_winsor_decile_summary.csv', index_col=0)
size_summary[cols[:-1]].style.format('{:.4f}')


In [ ]:
size_layers = pd.read_csv(DEMO / 'size_me_winsor_decile_layer_returns.csv', parse_dates=['date'])
fig = bt.plot_layers(size_layers)
fig.update_layout(title='Size/ME winsorized decile layer returns')
fig.show()


In [ ]:
size_spread = pd.read_csv(DEMO / 'size_me_winsor_decile_long_short.csv', parse_dates=['date'])
fig = bt.plot_performance(size_spread)
fig.update_layout(title='Size decile spread: D01 - D10')
fig.show()


In [ ]:
size_ic = pd.read_csv(DEMO / 'size_me_winsor_ic.csv', parse_dates=['date'])
fig = go.Figure()
for col in ['cum_ic', 'cum_rank_ic', 'cum_cos_ic']:
    fig.add_trace(go.Scatter(x=size_ic['date'].to_numpy(dtype='datetime64[ns]'), y=size_ic[col], name=col))
fig.update_layout(template='plotly_white', height=500, title='Size/ME cumulative IC')
fig.show()
display(size_ic[['date', 'ic', 'rank_ic', 'cos_ic', 'n_assets']].tail())
